# MosaicDiff — Google Colab

GPU-only notebook using the exact desktop pipeline. It never mounts Google Drive. Videos flow through `/input` and results through `/output`.

In [ ]:
#@title Configuration
HF_TOKEN = '' #@param {type:'string'}
DETECTOR_REPO = '' #@param {type:'string', placeholder:'owner/repo containing rfdetr.onnx'}
DETECTOR_FILE = 'rfdetr.onnx' #@param {type:'string'}
LORA_REPO = '' #@param {type:'string', placeholder:'owner/repo containing lora.safetensors'}
LORA_FILE = 'lora.safetensors' #@param {type:'string'}
+INPUT_VIDEO = '' #@param {type:'string', placeholder:'example.mp4; leave empty to process all files in /input'}
H3_SECONDS = 5 #@param {type:'slider', min:1, max:15, step:1}
H3_RESOLUTION = 800 #@param {type:'slider', min:512, max:1280, step:32}
COMPARE = False #@param {type:'boolean'}


In [ ]:
from pathlib import Path
import shutil

REPO = Path('/content/MosaicDiff')
INPUT = Path('/input')
OUTPUT = Path('/output')
MODELS = REPO / 'models'
if not REPO.exists():
    !git clone --depth 1 https://github.com/dotaku22/MosaicDiff.git /content/MosaicDiff
%cd /content/MosaicDiff
INPUT.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)
print('Input:', INPUT)
print('Output:', OUTPUT)


In [ ]:
# Colab's CUDA PyTorch is preserved; the desktop torch pin is not installed.
!pip -q install av 'numpy>=1.26' 'opencv-python<5' mmengine 'onnxruntime-gpu>=1.20' 'huggingface_hub>=0.26'
!if [ ! -d /content/ComfyUI ]; then git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI; fi
!pip -q install -r /content/ComfyUI/requirements.txt
!pip -q install --disable-pip-version-check --only-binary=:all: --extra-index-url https://pypi.nvidia.com nvidia-vfx
import torch
assert torch.cuda.is_available(), 'Enable Runtime > Change runtime type > T4 GPU (or another NVIDIA GPU)'
print(torch.cuda.get_device_name(0))


In [ ]:
#@title Hugging Face login and download all models
from mosaicdiff.colab import download_huggingface_models

if not HF_TOKEN:
    from huggingface_hub import notebook_login
    notebook_login()
download_huggingface_models(
    MODELS,
    token=HF_TOKEN,
    detector_repo=DETECTOR_REPO,
    detector_file=DETECTOR_FILE,
    lora_repo=LORA_REPO,
    lora_file=LORA_FILE,
)


In [ ]:
#@title Put videos in /input
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    (INPUT / Path(name).name).write_bytes(data)
print(*sorted(p.name for p in INPUT.iterdir()), sep='\n')


In [ ]:
#@title Process video
from mosaicdiff.colab import colab_settings, process_colab_videos

settings = colab_settings(
    comfy_root='/content/ComfyUI',
    model_directory=MODELS,
    output_directory=OUTPUT,
    h3_seconds=H3_SECONDS,
    h3_resolution=H3_RESOLUTION,
    compare=COMPARE,
)
if INPUT_VIDEO.strip():
    sources = [INPUT / Path(INPUT_VIDEO.strip()).name]
else:
    sources = sorted(p for p in INPUT.iterdir() if p.suffix.lower() in {'.mp4', '.mkv', '.mov', '.avi', '.webm', '.m4v'})
if not sources or not all(p.is_file() for p in sources):
    raise FileNotFoundError('Put a video in /input or set INPUT_VIDEO to its filename.')
outputs = process_colab_videos(sources, settings)
print('Finished:')
print(*outputs, sep='\n')


In [ ]:
#@title Inspect /output and download results
from google.colab import files

print(*sorted(p.name for p in OUTPUT.iterdir()), sep='\n')
for output in sorted(OUTPUT.iterdir()):
    if output.is_file() and output.suffix.lower() in {'.mp4', '.mkv', '.mov', '.avi', '.webm', '.m4v'}:
        files.download(str(output))
